In [ ]:
library(Seurat)
library(dplyr)
library(data.table)
library(ggplot2)
source("~/Projects/heads/clustering.r")

In [ ]:
data_dir = "/gpfs/gibbs/pi/braun/zy325"

In [ ]:
obj = readRDS(file.path(data_dir,"processed","theta1_dims50","scrcc_lymphoid_clustered_addCgenes.rds"))

# Put C genes back
Tcell = c(0:3,5,8,11,12,15)
B = c(9,13,16)
NK = c(6,10,14)
ILC = c(4)

# Contamination
# Hypoxic: SPP1+ VEGFA+ - 7
# Small clusters - 17

obj$lineage3 = case_when(
    obj$`RNA_snn_res.0.5` %in% Tcell~"T",
    obj$`RNA_snn_res.0.5` %in% B~"B",
    obj$`RNA_snn_res.0.5` %in% NK~"NK",
    obj$`RNA_snn_res.0.5` %in% ILC~"ILC",
    TRUE~"contamination")

In [ ]:
ilc2t = readRDS("scrcc_lymphoid_c4_clustered.rds")
ilc2t = subset(ilc2t, `RNA_snn_res.0.5` %in% c(1,3,4))
obj$lineage3[obj$name %in% colnames(ilc2t)] = "T"
obj = subset(obj,lineage3=="T")

In [ ]:
obj$lineage3_clusters = paste0("T_",obj$`RNA_snn_res.0.5`)
obj$lineage3_clusters[match(ilc2t$name,obj$name)] = paste0("ILC2T_",ilc2t$`RNA_snn_res.0.5`)

In [ ]:
# Recover AIR assay back to RNA
counts1 = LayerData(obj, assay = "RNA", layer = "counts")
counts2 = LayerData(obj, assay = "AIR", layer = "counts")
counts = rbind(counts1, counts2)
obj[["RNA"]] = CreateAssay5Object(counts = counts)
obj[["AIR"]] = NULL

In [ ]:
obj = clustering(obj,keep_c_genes = F,
                vars.to.regress = c("nFeature_RNA","nCount_RNA","percent.mt"),
                plot_QC_metrics = FALSE,
                group.by.vars = "batch_lab",
                harmony_theta=3,dims = 1:20,resolution=0.3)

In [ ]:
#saveRDS(obj,file="scrcc_t_ilc2t_clustered.rds")
#obj = readRDS("scrcc_lymphoid_c4_clustered.rds")

In [ ]:
obj = FindClusters(obj,resolution = 1)

In [ ]:
options(repr.plot.width=8,repr.plot.height=10)
VlnPlot(obj,features = c(
    "CD3D","CD3E","CD3G",#"TRAC","TRBC1","TRBC2","TRDC",
    "CD8A","CD8B","CD4",
    "NCAM1","FCGR3A",
    "IL7R","TBX21","IFNG",
    "GATA3","PTGDR2","IL1RL1",
    "KIT","RORC","IL23R"
   ),pt.size = 0,stack = TRUE,flip = TRUE)

In [ ]:
options(repr.plot.width=7,repr.plot.height=7)
DimPlot(obj,label=TRUE) + NoLegend()
#DimPlot(obj,label=TRUE,group.by = "lineage3_clusters") + NoLegend()

In [ ]:

options(repr.plot.width=7,repr.plot.height=7)
DimPlot(obj,cells.highlight = obj$name[obj$anno_cd8t == "CD8Tex_NMF3"],raster = T) + NoLegend()

In [ ]:
table(obj$lineage3_clusters,obj$seurat_clusters)

In [ ]:
### TCR mapping ###

tcrs = list.dirs("/gpfs/gibbs/project/braun/zy325/scrcc/raw/tcr_airrflow_output/cellranger",recursive = F)
tcrs = lapply(file.path(tcrs,"outs","filtered_contig_annotations.csv"),function(x){
    cr = fread(x) %>% mutate(
        sample_id2=gsub("^.*cellranger\\/","",gsub("\\/outs.*$","",x)),
        sample_barcode=paste0(sample_id2,'_',barcode))
    return(cr)
}) %>% rbindlist

trbs = tcrs %>% filter(chain == "TRB") 

obj$barcode = gsub("^.*removed_","",obj$name)
obj$sample_barcode = paste0(obj$sample_id2,"_",obj$barcode)
obj$wTCR = obj$sample_barcode %in% tcrs$sample_barcode
obj$wTRB = obj$sample_barcode %in% trbs$sample_barcode

options(repr.plot.width=15,repr.plot.height=7)
DimPlot(obj,group.by = "wTCR") | DimPlot(obj,group.by = "wTRB")

samples_missingtcr = paste0("SCRCC",c("14NORM","15","77","78"))

obj@meta.data %>% 
    filter(!sample_id2 %in% samples_missingtcr) %>%
    group_by(`RNA_snn_res.0.3`) %>%
    summarize(n=n(),
        n_wTCR=sum(wTCR),prop_wTCR=n_wTCR/n,
        n_wTRB=sum(wTRB),prop_wTRB=n_wTRB/n) %>% arrange(desc(prop_wTCR))

In [ ]:
6347+1518+179+58+6+129

In [ ]:
table(obj$`RNA_snn_res.0.3`[obj$anno_cd8t == "CD8Tex_NMF3"]) %>% sort(decreasing=TRUE) #%>% sum
table(obj$`RNA_snn_res.0.3`[obj$anno_cd8t == "CD8Tstr_HSPA6"]) %>% sort(decreasing=TRUE) #%>% sum
table(obj$`RNA_snn_res.0.3`[obj$anno_cd8t == "CD8Tn_CCR7"]) %>% sort(decreasing=TRUE) #%>% sum

In [ ]:
# 0,3,8
table(obj$`RNA_snn_res.0.3`[obj$anno_cd8t == "CD8Tex_NMF3"]) %>% sort(decreasing=TRUE) #%>% sum
table(obj$`RNA_snn_res.0.3`[obj$anno_cd8t == "CD8Tstr_HSPA6"]) %>% sort(decreasing=TRUE) #%>% sum
table(obj$`RNA_snn_res.0.3`[obj$anno_cd8t == "CD8Tn_CCR7"]) %>% sort(decreasing=TRUE) #%>% sum

In [ ]:
6043+1862+203+155+52+24

In [ ]:
# 0,2,7
table(obj$`RNA_snn_res.0.3`[obj$anno_cd8t == "CD8Tex_NMF3"]) %>% sort(decreasing=TRUE) #%>% sum
table(obj$`RNA_snn_res.0.3`[obj$anno_cd8t == "CD8Tstr_HSPA6"]) %>% sort(decreasing=TRUE) #%>% sum
table(obj$`RNA_snn_res.0.3`[obj$anno_cd8t == "CD8Tn_CCR7"]) %>% sort(decreasing=TRUE) #%>% sum


In [ ]:
# 0,3,5, 8?
table(obj$`RNA_snn_res.0.3`[obj$anno_cd8t == "CD8Tex_NMF3"]) %>% sort(decreasing=TRUE) #%>% sum
table(obj$`RNA_snn_res.0.3`[obj$anno_cd8t == "CD8Tstr_HSPA6"]) %>% sort(decreasing=TRUE) #%>% sum
table(obj$`RNA_snn_res.0.3`[obj$anno_cd8t == "CD8Tn_CCR7"]) %>% sort(decreasing=TRUE) #%>% sum


In [ ]:
# 1,2,7
table(obj$`RNA_snn_res.0.3`[obj$anno_cd8t == "CD8Tex_NMF3"]) %>% sort(decreasing=TRUE) #%>% sum
table(obj$`RNA_snn_res.0.3`[obj$anno_cd8t == "CD8Tstr_HSPA6"]) %>% sort(decreasing=TRUE) #%>% sum
table(obj$`RNA_snn_res.0.3`[obj$anno_cd8t == "CD8Tn_CCR7"]) %>% sort(decreasing=TRUE) #%>% sum


In [ ]:

options(repr.plot.width=8.5,repr.plot.height=7)
DimPlot(obj,cells.highlight = obj$name[obj$anno_cd8t == "CD8Tex_NMF3"],raster = T)#,sizes.highlight = .1,alpha = 1,pt.size = .1)

DimPlot(obj,cells.highlight = colnames(ilc2t),raster = T)

In [ ]:
options(repr.plot.width=16,repr.plot.height=8)

DimPlot(obj,group.by = "batch_lab") | DimPlot(obj,group.by = "batch_seq_rna") #+ NoLegend()

In [ ]:
# Put C genes back
cd8t = c(1,2,4,5,7,10)#c(1,3,4,6,8)
cd4t = c(0,3,8) #c(0,2,7)
cyclingt = c(6,9) #c(5)

obj$lineage3.5 = case_when(
    obj$`RNA_snn_res.0.3` %in% cd8t ~ "CD8T",
    obj$`RNA_snn_res.0.3` %in% cd4t ~ "CD4T",
    obj$`RNA_snn_res.0.3` %in% cyclingt  ~ "CyclingT",
    TRUE~"contamination")

options(repr.plot.width=16,repr.plot.height=8)

DimPlot(obj,group.by = "lineage3_clusters") | DimPlot(obj,group.by = "lineage3.5") #+ NoLegend()

In [ ]:
options(repr.plot.width=15,repr.plot.height=10)
VlnPlot(obj,features = c(
    "CD3D","CD3E","CD3G",
    "TRBC1","TRBC2","TRAC",
    "CD8A","CD8B","CD4","CD69","FOXP3",
    "CD79A","CD79B","MS4A1","MZB1","JCHAIN",
    "NCAM1","NCR1","FCGR3A",
    "MKI67","TOP2A",
    "SPP1","VEGFA",
    "CD68","S100A9","FCN1","C1QC",
    "HBB","PPBP","EPCAM","ALDOB","PECAM1","COL1A1","PTPRC"),pt.size = 0,stack = TRUE,flip = TRUE)

In [ ]:
options(repr.plot.width=10,repr.plot.height=10)
obj %>% 
VlnPlot(fill.by = "ident",features = c(
    "CD3D","CD3E","CD3G","CD8A","CD8B",
    "FOXP3","IKZF2","CTLA4",
    "CXCL13","PDCD1","HAVCR2",
    "IFNG","MX1","ISG15",
    "TOX","ZNF683","ITGA1",
    "NR4A1","LMNA","DNAJB1","HSPA1A",
    "SLC4A10","KLRB1","IL7R",
    "FCGR3A","PRF1","GZMB"),pt.size = 0,stack = T,flip = T)

In [ ]:
options(repr.plot.width=15,repr.plot.height=5)
FeaturePlot(obj,features = c("ZNF683","CCR7","GZMK"),ncol = 3,raster = T)

In [ ]:
options(repr.plot.width=10,repr.plot.height=10)
obj %>% 
VlnPlot(fill.by = "ident",#group.by = "lineage4",
        features = c(
            "CD3D","CD3E","CD3G","CD8A","CD8B",
            "THEMIS","RUNX1",
            "NR4A1","LMNA",
            "CX3CR1","GZMB","FGFBP2",  
            "GZMK","TNF","CCL4",
            "TNFRSF9","PDCD1","HAVCR2",
            "DNAJA1","HSPA6",
            "MX1","ISG15",
            "IL7R","LEF1","GPR183","FOXP1",
            "ZNF683","ITGAE","XCL1",
            "KIT","IL23R",
            "SLC4A10","KLRB1"),pt.size = 0,stack = T,flip = T)


options(repr.plot.width=10,repr.plot.height=10)
obj %>% 
DotPlot(#group.by = "lineage4",
        cols = "PiYG",
        features = c(
            "CD3D","CD3E","CD3G","CD8A","CD8B",
            "THEMIS","RUNX1",
            "NR4A1","LMNA",
            "CX3CR1","GZMB","FGFBP2",  
            "GZMK","TNF","CCL4",
            "TNFRSF9","PDCD1","HAVCR2",
            "DNAJA1","HSPA6",
            "MX1","ISG15",
            "IL7R","LEF1","GPR183","FOXP1",
            "ZNF683","ITGAE","XCL1",
            "KIT","IL23R",
            "SLC4A10","KLRB1")) + coord_flip()+theme(axis.text.x = element_text(angle = 45,hjust = 1))

In [ ]:
# Put C genes back
tex = c(0,1,6,7,10,16)


obj$lineage4 = case_when(
    obj$`RNA_snn_res.1` %in% tex ~ "CD8Tex_PDCD1",
    obj$`RNA_snn_res.1` == 11 ~ "CD8Tisg_MX1",
    obj$`RNA_snn_res.1` == 9  ~ "CD8Trm_ZNF683",
    obj$`RNA_snn_res.1` == 8  ~ "CD8Teff_GZMK", # GZMK, TNF, CCL4
    obj$`RNA_snn_res.1` %in% c(3,12)  ~ "CD8Teff_CX3CR1", # CX3CR1, FGFBP2, S1PR1
    obj$`RNA_snn_res.1` == 5  ~ "CD8Tnl_LEF1",
    obj$`RNA_snn_res.1` == 13  ~ "CD8Thsp_HSPA6",
    obj$`RNA_snn_res.1` == 2  ~ "MAIT_SLC4A10",
    obj$`RNA_snn_res.1` %in% c(4)  ~ "CD8Tea_NR4A1",
    obj$`RNA_snn_res.1` == 14  ~ "CD8T_THEMIS",
    obj$`RNA_snn_res.1` == 15  ~ "ILC3_IL23R",
    TRUE~"contamination")

In [ ]:
#obj = RunTSNE(obj,reduction = "harmony",dims = 1:20)
obj = RunUMAP(obj,reduction ="harmony",dims=1:20)#,n.neighbors = 50,min.dist=0.1)

In [ ]:
"TRAC" %in% rownames(obj@assays$AIR@counts)

In [ ]:
options(repr.plot.width=7,repr.plot.height=7)
DimPlot(obj,group.by = "lineage4",alpha = .75,pt.size = .1,cols=c(get_palette("Set1",9),"grey90","black")) #+ NoLegend()

In [ ]:
saveRDS(subset(obj,lineage4 == "CD8Tex_PDCD1"),file="scrcc_t_ilc2t_tex.rds")

In [ ]:
library(ggsankey)
options(repr.plot.width=7,repr.plot.height=7)

obj@meta.data %>% 
    make_long(lineage4,anno3) %>%
    ggplot(aes(x = x, 
               next_x = next_x, 
               node = node, 
               next_node = next_node,
               fill = factor(node),
               label = node)) +
    geom_sankey(flow.alpha = 0.5, node.color = 1) +
    geom_sankey_label(size = 3.5, color = 1, fill = "white") +
    theme_sankey(base_size = 16) + NoLegend()

In [ ]:
library(ggsankey)

obj@meta.data %>% 
    filter(!is.na(anno_cd8t)) %>%
    make_long(lineage4,anno_cd8t) %>% # 
    ggplot(aes(x = x, 
               next_x = next_x, 
               node = node, 
               next_node = next_node,
               fill = factor(node),
               label = node)) +
    geom_sankey(flow.alpha = 0.5, node.color = 1) +
    geom_sankey_label(size = 3, color = 1, fill = "white") +
    theme_sankey(base_size = 16) + NoLegend()

In [ ]:
table(obj$lineage3.5[obj$anno_cd8t == "CD8Tex_NMF3"]) %>% sort(decreasing = TRUE)
table(obj$lineage3.5[obj$anno_cd8t == "CD8Tstr_HSPA6"]) %>% sort(decreasing = TRUE)

In [ ]:
table(obj$lineage3.5[obj$anno_cd8t == "CD8Tex_NMF3"]) %>% sort(decreasing = TRUE)
table(obj$lineage3.5[obj$anno_cd8t == "CD8Tstr_HSPA6"]) %>% sort(decreasing = TRUE)
table(obj$lineage3.5[obj$anno_cd8t == "CD8Tn_CCR7"]) %>% sort(decreasing = TRUE)

In [ ]:
table(obj$lineage3.5[obj$anno_cd8t == "CD8Tex_NMF3"]) %>% sort(decreasing = TRUE)
table(obj$lineage3.5[obj$anno_cd8t == "CD8Tstr_HSPA6"]) %>% sort(decreasing = TRUE)
table(obj$lineage3.5[obj$anno_cd8t == "CD8Tn_CCR7"]) %>% sort(decreasing = TRUE)

In [ ]:
m = FindMarkers(obj,`ident.1` = c(2),only.pos = F,logfc.threshold = .6)
m %>% filter(p_val_adj<0.05) %>% arrange(desc(avg_log2FC)) %>% filter(abs(pct.1-pct.2)>.08)

In [ ]:
m = FindMarkers(obj,`ident.1` = "CD8Tnl_LEF1",group.by = "lineage4",only.pos = TRUE,logfc.threshold = .5)
m %>% filter(p_val_adj<0.05) %>% arrange(desc(avg_log2FC)) %>% filter(abs(pct.1-pct.2)>.08)

In [ ]:
###### Validation from CD8T previous annotation ######

cd8t = readRDS("/gpfs/gibbs/project/braun/zy325/scrcc/processed/seurat_objects/Clustering_CD8T_0912.rds")

In [ ]:
cd8t$nCount_RNA_log = log(cd8t$nCount_RNA)
cd8t$nFeature_RNA_log = log(cd8t$nFeature_RNA)

In [ ]:
options(repr.plot.width=10,repr.plot.height=6)
VlnPlot(cd8t,features = c("nCount_RNA","nFeature_RNA","percent.mt"),pt.size = 0,group.by = "Round2_cls",stack = TRUE,flip = TRUE,layer = "data")

In [ ]:
options(repr.plot.width=10,repr.plot.height=6)
VlnPlot(cd8t,features = c("nCount_RNA_log","nFeature_RNA_log"),pt.size = 0,group.by = "Round2_cls",stack = TRUE,flip = TRUE,layer = "data")

In [ ]:
### TCR mapping ###
tcrs = list.dirs("/gpfs/gibbs/project/braun/zy325/scrcc/raw/tcr_airrflow_output/cellranger",recursive = F)
tcrs = lapply(file.path(tcrs,"outs","filtered_contig_annotations.csv"),function(x){
    cr = fread(x) %>% mutate(
        sample_id2=gsub("^.*cellranger\\/","",gsub("\\/outs.*$","",x)),
        sample_barcode=paste0(sample_id2,'_',barcode))
    return(cr)
}) %>% rbindlist

cd8t$barcode = gsub("^.*removed_","",cd8t$name)
cd8t$sample_barcode = paste0(cd8t$sample,"_",cd8t$barcode)
cd8t$wTCR = cd8t$sample_barcode %in% tcrs$sample_barcode

options(repr.plot.width=8,repr.plot.height=7)
DimPlot(cd8t,group.by = "wTCR") #+ NoLegend()

In [ ]:
samples_missingtcr = paste0("SCRCC",c("14NORM","15","77","78"))

cd8t@meta.data %>% 
    filter(!sample %in% samples_missingtcr) %>%
    group_by(Round2_cls) %>%
    summarize(n=n(),
        n_wTCR=sum(wTCR),prop_wTCR=n_wTCR/n) %>% arrange(desc(prop_wTCR))